<div align="center">
  <img src="https://raw.githubusercontent.com/maxhormazabal/summer-school-agentic-rag/refs/heads/main/tutorial_imgs/logo_red.svg" alt="Summer School DAG Logo" width="200"/>
</div>

# Agentic RAG with Vision AI & Knowledge Graphs

We build an end-to-end **Agentic RAG** system over a knowledge graph populated from PDF match
reports using vision AI — in two parts:

- **Part 1 — Interactive Lab.** A small, controlled setting (3 reports) where *you* complete the
  code for each phase: ontology, extraction checks, graph queries, and the agent.
- **Part 2 — At Scale.** The same pipeline running over the full official dataset of **1793**
  reports, with extraction already pre-computed.

Both phases use the **OpenAI API** (no GPU, no model downloads): `gpt-4o` reads the report
images, and `gpt-4o-mini` drives the agent.

| Stage | What happens |
|---|---|
| **1 — Ontology** | Define the domain as Pydantic models |
| **2 — VLM Extraction** | `gpt-4o` (vision) reads PDF pages → structured JSON |
| **3 — Knowledge Graph** | JSON → Neo4J nodes & relationships (MERGE, idempotent) |
| **4 — Agentic RAG** | Natural language → Cypher → answer via tool-calling (`gpt-4o-mini`) |


In [ ]:
import base64
import requests, io
from IPython.display import SVG, display

graph = """
flowchart LR
    A[PDF] --> B["Vision Language Model"]
    B --> C["Structured JSON<br/>(Pydantic)"]
    C --> D["Neo4J Graph<br/>(MERGE)"]
    D --> E[Agent]
    E --> F[Answer]
"""

graphbytes = graph.encode("ascii")
base64_bytes = base64.b64encode(graphbytes)
base64_string = base64_bytes.decode("ascii")
url = f"https://mermaid.ink/svg/{base64_string}"

svg_data = requests.get(url).text

display(SVG(svg_data))

> **Domain:** Official match reports (*actes*) from the Federació Catalana de Futbol (FCF), written in Catalan.

> ✅ **Part 1 — Solutions.** This notebook is the interactive lab with every **📝 Your turn**
> exercise already filled in. Use it to check your work or to run Part 1 end-to-end without
> editing anything. (Part 2 has no exercises, so it's omitted here — see the main notebook.)

## 0 · Setup

Credentials, models, dependencies, connectivity — shared by both parts.

> ⏳ For **Part 2**, this section downloads the pre-computed 1793-report dataset (~500 MB,
> one-time, cached after the first run). Extraction and the agent call the **OpenAI API**, so
> no GPU is required.


### Credentials

You need two things, both free:

1. A **Neo4J AuraDB** instance ([console.neo4j.io](https://console.neo4j.io)) — paste its URI,
   user and password below.
2. An **OpenAI API key** ([platform.openai.com](https://platform.openai.com/api-keys)) — the
   extraction (vision) and the agent both call OpenAI. In Colab, the cleanest way is to store it
   as a **Secret** (🔑 in the left sidebar) named `OPENAI_API_KEY`; otherwise paste it below or
   you'll be prompted.


In [ ]:
import os, getpass

# ── Neo4J AuraDB (free) — paste yours ─────────────────────────────────────────
NEO4J_URI      = '<paste-your-neo4j-uri>'        # e.g. neo4j+s://<id>.databases.neo4j.io
NEO4J_USERNAME = '<paste-your-neo4j-user>'       # default: neo4j
NEO4J_PASSWORD = '<paste-your-neo4j-password>'
NEO4J_DATABASE = '<paste-your-neo4j-database>'    # usually 'neo4j'; leave as-is to use the server default
# ──────────────────────────────────────────────────────────────────────────────

# Only export non-placeholder values, so a local .env (or Colab Secrets) is preserved.
for _k, _v in {
    "NEO4J_URI": NEO4J_URI, "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD, "NEO4J_DATABASE": NEO4J_DATABASE,
}.items():
    if _v and not _v.startswith("<"):
        os.environ[_k] = _v

# ── OpenAI API key ────────────────────────────────────────────────────────────
OPENAI_API_KEY = ''   # leave empty to use Colab Secrets, an existing env var, or a prompt
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
elif not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("Credentials set ✓")

### Models — OpenAI GPT

This system is **backend-agnostic** (`src/common/llm.py`): each role independently targets a
backend by environment variable. Here both roles use OpenAI.

| Role | Model | Why |
|---|---|---|
| **Extraction** | `gpt-4o` | Reads the report *image* → JSON. Needs vision + strict structured output. |
| **Agent** | `gpt-4o-mini` | Turns questions into Cypher via tool-calls. Light, fast, cheap. |

> No GPU and no model downloads — the calls go to the OpenAI API. Extraction runs only on a few
> sample reports (Part 1); the 1793-report dataset (Part 2) is pre-computed, so total OpenAI
> usage stays tiny.


In [ ]:
# ── Role → OpenAI model ───────────────────────────────────────────────────────
import os

os.environ["EXTRACTION_BACKEND"] = "openai"
os.environ["EXTRACTION_MODEL"]   = "gpt-4o"        # vision: report image → JSON
os.environ["AGENT_BACKEND"]      = "openai"
os.environ["AGENT_MODEL"]        = "gpt-4o-mini"   # text: question → Cypher (tool-calls)

# Drop any leftover local-server overrides so nothing routes to a local endpoint.
for _k in ("LOCAL_VLLM_BASE_URL", "LOCAL_VLLM_MODEL", "LOCAL_VLLM_API_KEY"):
    os.environ.pop(_k, None)

print("Extraction → openai ·", os.environ["EXTRACTION_MODEL"])
print("Agent      → openai ·", os.environ["AGENT_MODEL"])

In [ ]:
import sys, os, subprocess

# ── Environment detection ─────────────────────────────────────────────────────
try:
    from google.colab import userdata  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_SIMULATE = os.environ.get("COLAB_SIMULATE", "").lower() in ("1", "true")
GITHUB_REPO_URL = "https://github.com/maxhormazabal/summer-school-agentic-rag.git"

if IN_COLAB:
    REPO_NAME = GITHUB_REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL], check=True)
    os.chdir(REPO_NAME)
    # System deps for the pipeline: poppler (PDF→image) + graphviz (ontology diagram).
    subprocess.run(
        ["apt-get", "install", "-y", "-q", "poppler-utils", "graphviz"],
        capture_output=True, check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "openai"], check=True)
elif COLAB_SIMULATE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

mode = "Google Colab" if IN_COLAB else ("Colab Simulation" if COLAB_SIMULATE else "Local Dev")
print(f"Environment : {mode}")
print(f"Repo root   : {REPO_ROOT}")

### Connectivity check

Confirm both services are reachable before going further: a one-token ping to OpenAI and a
`RETURN 1` against Neo4J.


In [ ]:
from rich.table import Table
from rich.console import Console
from src.graph.neo4j_client import Neo4jClient
from src.common.llm import get_client_and_model

console = Console()
t = Table(title="Connectivity")
for col in ("Service", "Status", "Detail"):
    t.add_column(col)

ok = True

# OpenAI — a tiny chat call validates the key and the agent model.
try:
    _client, _model = get_client_and_model("agent")
    _client.chat.completions.create(
        model=_model, messages=[{"role": "user", "content": "ping"}], max_tokens=5,
    )
    t.add_row("OpenAI", "✅", _model)
except Exception as e:
    t.add_row("OpenAI", "❌", str(e)[:70]); ok = False

# Neo4J
try:
    with Neo4jClient() as c:
        c.run_read("RETURN 1 AS x")
    t.add_row("Neo4J", "✅", "OK")
except Exception as e:
    t.add_row("Neo4J", "❌", str(e)[:70]); ok = False

console.print(t)
assert ok, "Fix connectivity before continuing."

---
# Part 1 — Interactive Lab

Hands-on with **3 sample match reports** — small enough to *touch things* and see what happens.

> ⚠️ **How to use Part 1**
> - Edit **only** the cells marked **📝 Your turn**. Everything else is scaffolding.
> - Each 📝 exercise is followed by a **✅ / ❌ check** cell.
> - Broke something? **Re-run from the top.** Nothing here is destructive.

Part 1 uses the 3 reports shipped in the repo (`data/`), so no extra download is needed.

## 1 · The domain — FCF match reports

Each report is a single PDF page (in Catalan) containing the lineups (**TITULARS** / **SUPLENTS**),
technical staff, goals (**GOLS**), cards (**TARGETES**), stadium and referee. The VLM reads the
**image** — no OCR. Let's look at one.

In [ ]:
from pdf2image import convert_from_path
from IPython.display import display
from src.common.paths import DOCS_DIR

page = convert_from_path(str(DOCS_DIR / "example1.pdf"), dpi=110, first_page=1, last_page=1)[0]
display(page.resize((int(page.width * 0.6), int(page.height * 0.6))))

## 2 · Ontology

The ontology is just a set of **Pydantic models** describing what a match report contains.
`MatchExtraction` is the root; it nests `Team`, `Player`, `Goal`, `Card`, `Stadium`, `Referee`.
Here is the diagram:

In [ ]:
from src.ontology.visualize import render_ontology
render_ontology()   # writes out/ontology.png and returns a displayable diagram

### 📝 Your turn — complete the `Goal` model

A goal in the report has a minute, the running scoreline, who scored, which side scored, and
its type. **Fill in the fields** of `MyGoal` below to match this spec:

| field | type |
|---|---|
| `minute` | `int` |
| `scoreline_home` | `int` |
| `scoreline_away` | `int` |
| `scorer_name` | `str` |
| `scoring_team` | `Literal["home", "away"]` |
| `type` | `GoalType` (enum: `regular` / `own` / `penalty`) |

In [ ]:
from pydantic import BaseModel
from typing import Literal
from src.ontology.schema import GoalType   # provided enum: regular | own | penalty

class MyGoal(BaseModel):
    minute: int
    scoreline_home: int
    scoreline_away: int
    scorer_name: str
    scoring_team: Literal["home", "away"]
    type: GoalType

In [ ]:
# ✅ check
expected = {"minute", "scoreline_home", "scoreline_away", "scorer_name", "scoring_team", "type"}
sample = {"minute": 23, "scoreline_home": 1, "scoreline_away": 0,
          "scorer_name": "PUIG, ANNA", "scoring_team": "home", "type": "regular"}
try:
    g = MyGoal(**sample)
    got = set(MyGoal.model_fields)
    assert got == expected, f"Fields differ.\n  expected: {expected}\n  got:      {got}"
    assert g.type == GoalType.regular and g.scoring_team == "home"
    print("✅ Your Goal model has exactly the right fields and validates the sample.")
except Exception as e:
    print("❌", type(e).__name__, "-", e)

## 3 · Extraction (VLM → JSON)

The extractor sends each report **image** to `gpt-4o` and asks for JSON that validates against
our ontology — no OCR, no layout parsing. Let's look at the domain prompt it uses first.


The VLM receives the page image plus this domain prompt and must return JSON validated against
the ontology:

In [ ]:
from src.extraction.vlm_extractor import _SYSTEM_PROMPT
print(_SYSTEM_PROMPT)

Now run extraction **live** on all 3 sample reports. Each call sends the page image to
`gpt-4o` and validates the JSON against the ontology (with a retry on failure). Results are
written to `data/extracted/` and feed the graph below.


In [ ]:
from pathlib import Path
from src.extraction.pdf_to_images import convert_all
from src.extraction.vlm_extractor import extract
from src.common.paths import IMAGES_DIR, EXTRACTED_DIR
from src.ontology.schema import MatchExtraction

convert_all()                       # data/documents/example*.pdf → data/images/example*.png
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

for png in sorted(IMAGES_DIR.glob("example*.png")):
    try:
        m = extract(png)
        (EXTRACTED_DIR / f"{png.stem}.json").write_text(
            m.model_dump_json(indent=2), encoding="utf-8")
        print(f"  {png.name}: {m.home.name} {m.score_home}-{m.score_away} {m.away.name}"
              f" · goals={len(m.goals)}")
    except Exception as e:
        print(f"  {png.name}: extraction failed ({type(e).__name__}) — keeping any cached JSON.")

### 📝 Your turn — write a data-quality check

When you extract data automatically, you need cheap checks that it isn't hallucinated. Implement
`quality_check(m)` returning a dict with two booleans:

- **`goals_ok`** — the number of goals equals the final score: `len(m.goals) == m.score_home + m.score_away`
- **`scorers_ok`** — every goal's scorer appears in *some* lineup, after name normalisation.

Helpers: `normalize_name(text)`; lineups are `m.home.lineup` / `m.away.lineup`; each entry has
`entry.player.name`.

In [ ]:
from src.common.ids import normalize_name
from src.ontology.schema import MatchExtraction

def quality_check(m: MatchExtraction) -> dict:
    names = {normalize_name(e.player.name) for e in (m.home.lineup + m.away.lineup)}
    goals_ok = len(m.goals) == m.score_home + m.score_away
    scorers_ok = all(normalize_name(g.scorer_name) in names for g in m.goals)
    return {"goals_ok": goals_ok, "scorers_ok": scorers_ok}

In [ ]:
# ✅ check — runs your function over the 3 extracted reports
from pathlib import Path
from src.common.paths import EXTRACTED_DIR

all_ok = True
files = sorted(EXTRACTED_DIR.glob("example*.json"))
for f in files:
    m = MatchExtraction.model_validate_json(f.read_text(encoding="utf-8"))
    r = quality_check(m) or {}
    print(f"  {f.name}: goals_ok={r.get('goals_ok')}  scorers_ok={r.get('scorers_ok')}")
    all_ok = all_ok and bool(r.get("goals_ok")) and bool(r.get("scorers_ok"))
print("✅ quality_check passes on all reports." if (files and all_ok)
      else "❌ Not all checks pass — review your quality_check (or the live extraction above).")

## 4 · The knowledge graph

We turn the JSON into Neo4J nodes and relationships. Ingestion uses `MERGE`, so it is
**idempotent** — running it twice doesn't duplicate anything. Let's load the 3 reports.

In [ ]:
from src.graph.constraints import apply as apply_constraints
from src.graph.ingest import ingest_all

apply_constraints()
counts = ingest_all()   # ingests the 3 reports from data/extracted/
print("Node counts:", counts)

### 📝 Your turn — count goals per team in Cypher

The graph has `(Goal)-[:FOR_TEAM]->(Team)` and every `Team` has a `name` property. **Complete
the query** so it returns one row per team with its goal count. Replace `___` with the right
aggregation.

In [ ]:
from src.agent.tools import run_cypher

cypher = """
MATCH (g:Goal)-[:FOR_TEAM]->(t:Team)
RETURN t.name AS team, count(g) AS goals
ORDER BY goals DESC
"""

result = run_cypher(cypher)
print(result)

In [ ]:
# ✅ check — compares your query against the count computed in Python from the JSONs
from collections import Counter
from src.common.paths import EXTRACTED_DIR

expected = Counter()
for f in sorted(EXTRACTED_DIR.glob("example*.json")):
    m = MatchExtraction.model_validate_json(f.read_text(encoding="utf-8"))
    for goal in m.goals:
        team = m.home.name if goal.scoring_team == "home" else m.away.name
        expected[team] += 1

if result.get("error"):
    print("❌ Query error:", result["error"])
else:
    got = {row["team"]: row["goals"] for row in result["rows"]}
    print("expected  :", dict(expected))
    print("your query:", got)
    print("✅ Your query matches the Python aggregate!" if got == dict(expected)
          else "❌ Counts differ — check your aggregation (and that you grouped by team).")

## 5 · The agent

The agent turns a natural-language question into Cypher using **tool-calling**: it asks for the
schema, validates a query, runs it, and paraphrases the answer — recovering from errors along
the way. Watch one run:

In [ ]:
from src.agent.agent import ask

result = ask("What was the score between Cirera and L'Estartit?")
print("\nAnswer:", result.answer)
print("Cypher attempts:")
for i, a in enumerate(result.cypher_attempts, 1):
    print(f"  [{i}] ok={a['ok']} rows={a['rows']}  {a['query'][:70].strip()}")

### 📝 Your turn (A) — ask your own question

Replace the question with one of your own about the 3 matches (scores, scorers, cards, stadiums,
referees…). Run it and read the Cypher the agent generated above the answer.

In [ ]:
MY_QUESTION = "Who scored in the match between Cirera and L'Estartit, and in which minute?"

result = ask(MY_QUESTION)
print("\nAnswer:", result.answer)

In [ ]:
# ✅ check
assert result.answer and result.answer.strip(), "❌ The agent returned an empty answer."
print("✅ The agent answered. Scroll up to see the Cypher it wrote and ran.")

### 📝 Your turn (B) — change the agent's tone

The agent's behaviour comes from a **system prompt**. You can append one instruction to it at
runtime (no file editing) and see how the answer changes. Try a very formal tone, a playful one,
or "always answer in Catalan".

In [ ]:
from src.agent import prompts

TONE_INSTRUCTION = "Always answer in Catalan, in a friendly and concise tone."

# Capture the base prompt once so re-running this cell with a new tone doesn't stack rules.
if not hasattr(prompts, "_BASE_SYSTEM_PROMPT"):
    prompts._BASE_SYSTEM_PROMPT = prompts.SYSTEM_PROMPT
prompts.SYSTEM_PROMPT = prompts._BASE_SYSTEM_PROMPT + "\n\nADDITIONAL STYLE RULE: " + TONE_INSTRUCTION

result = ask("What was the score between Cirera and L'Estartit?")
print("\nAnswer:", result.answer)

In [ ]:
# ✅ check
assert result.answer and result.answer.strip(), "❌ Empty answer."
print("✅ Compare this answer's tone with the one from section 5. Same facts, different style.")
print("   (Re-run the cell above with a different TONE_INSTRUCTION to experiment.)")